Save base Logit, KNN, SVC, LogisticRegression, CatBoost, XGboost with pipeline. Sedimentation_Stk feature was used!

Based on `research_14_SMOTE_influence_on_metrics.ipynb` it was decided to use SMOTE for all models, except Logit!

# Import libraries

In [1]:
# Changes to all modules will automatically be applied when any cell runs. 
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

import joblib

from statsmodels.api import Logit
from statsmodels.api import Logit # type: ignore
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier

from sklearn.metrics import (
    make_scorer,
    f1_score,
)

import optuna
from functools import partial

from pathlib import Path
import sys

from typing import Type
from sklearn.base import BaseEstimator

sys.path.append(
    str(Path('..', 'utils_functionality', 'split_utils'))
)
sys.path.append(
    str(Path('..', 'utils_functionality', 'models'))
)

from split_tools import get_train_test
from modelling4_utils import (
    # _prepare_df,
    MLPipeline,
    StatsModelsEstimator,
    update_estimator_params,
    OptunaOptimizer,
    GridSearchOptimizer,
    RANDOM_STATE,
)

seed = RANDOM_STATE

In [3]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Settings

In [4]:
model_postfix = 'opt-smote_opt-model'
add_smote = True
is_smotenc = False
common_smote_params = {
    # 'categorical_features': (
    #     'wettability',
    #     'inclination',
    # ),
}
scoring_metrics = {"f1_macro": make_scorer(f1_score, average="macro"),}
step_scoring_average = "mean"
n_trials = 2500 # was checked with 1000 and 5000

save_model_and_metrics = True
metrics_file = "metrics_modelling4_4.1-opt_smote_no_ensembles.xlsx"

smote_params = {
    'splashing': {
        **common_smote_params,
        'k_neighbors': 7,
        'sampling_strategy': 0.9, # 0.8 for SMOTENC
    },
    'no_fragmentation': {
        **common_smote_params,
        'k_neighbors': 5,
        'sampling_strategy': 0.6,
    }
}

## Optimization functions

In [5]:
def optimize_estimator_optuna(
    target:str,
    estimator:Type[BaseEstimator],
    estimator_params:dict,
    objective:callable,
    n_trials:int,
    step_scoring_average:str=step_scoring_average,
    params_to_process:list=None,
    model_postfix:str=model_postfix,
    add_smote:bool=add_smote,
    is_smotenc:bool=is_smotenc,
    smote_params:dict=smote_params,
    scoring_metrics:dict=scoring_metrics,
    save_model_and_metrics:bool=save_model_and_metrics,
    metrics_file:str=metrics_file,
    seed:int=seed,
):
    
    target_smote_params = smote_params[target]
    
    estimator_params = estimator_params.copy()
    
    # Switch off probability for SVC, since it is long to optimize
    if 'probability' in estimator_params:
        estimator_params['probability'] = False
    
    ml_pipe = MLPipeline(
        target=target,
        estimator=estimator,
        estimator_params=estimator_params,
        model_postfix=model_postfix,
        add_smote=add_smote,
        is_smotenc=is_smotenc,
        smote_params=target_smote_params,
        scoring_metrics=scoring_metrics,
        step_scoring_average=step_scoring_average,
    )
    
    opt = OptunaOptimizer(
        objective=partial(objective, ml_pipe=ml_pipe),
        study_name="model_study",
        direction="maximize",
        seed=seed,
    )
    
    opt.optimize(n_trials=n_trials)
    
    best_params = opt.study.best_params
    
    if params_to_process:
        for param in params_to_process:
            # Find one key in best_params which contains param
            key = next((k for k in best_params.keys() if param in k), None)
            if key:
                # Remove gamma_type if it is float
                if (key=='gamma_type') and (best_params[key]=='float'):
                    best_params.pop(key)
                    continue
                        
                best_params[param] = best_params.pop(key)

    print('best_params')
    display(best_params)
    
    # Switch on probability for SVC, to get metrics like roc_auc for the final model
    if 'probability' in estimator_params:
        estimator_params['probability'] = True
    
    ml_pipe = MLPipeline(
        target=target,
        estimator=estimator,
        estimator_params={**estimator_params, **best_params},
        model_postfix=model_postfix,
        add_smote=add_smote,
        is_smotenc=is_smotenc,
        smote_params=target_smote_params,
        step_scoring_average=step_scoring_average, # No need to pass, it would not be used
        metrics_file=metrics_file,
    )
    
    ml_pipe.run(
        save_model_and_metrics=save_model_and_metrics,
    )

# Logistic Regression

In [6]:
estimator = LogisticRegression
target = 'splashing'
estimator_params = {
    "fit_intercept": False,
    'max_iter': 1000,
}
params_to_process = ['penalty']

def logreg_objective(trial:optuna.trial.Trial, ml_pipe:MLPipeline):
    
    # # NOTE: Does not work, since optuna requires static parameters ranges
    # solver = trial.suggest_categorical('solver', ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'])
    # penalty_options = {
    #     'lbfgs': ['l2'],
    #     'liblinear': ['l1', 'l2'],
    #     'newton-cg': ['l2'],
    #     'newton-cholesky': ['l2'],
    #     'sag': ['l2'],
    #     'saga': ['l1', 'l2', 'elasticnet'],
    # }
    # penalty = trial.suggest_categorical('penalty', penalty_options[solver])
    # NOTE: warnings + inconvenient to create best model
    # solver_penalty_combinations = [
    #     ("liblinear", "l1"),
    #     ("liblinear", "l2"),
    #     ("saga", "l1"),
    #     ("saga", "l2"),
    #     ("saga", "elasticnet"),
    #     ("lbfgs", "l2"),
    #     ("newton-cg", "l2"),
    #     ("newton-cholesky", "l2"),
    #     ("sag", "l2"),
    # ]
    # solver, penalty = trial.suggest_categorical(
    #     'solver_penalty', solver_penalty_combinations
    # )
    
    solver = trial.suggest_categorical('solver', ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'])
    # solver = trial.suggest_categorical('solver', ['lbfgs',])
    
    if solver == 'liblinear':
        penalty = trial.suggest_categorical('penalty_liblinear', ['l1', 'l2'])
    elif solver == 'saga':
        penalty = trial.suggest_categorical('penalty_saga', ['l1', 'l2', 'elasticnet'])
    else:
        penalty = trial.suggest_categorical('penalty_other', ['l2'])
    
    # If C is large, the penalty should be small
    C = trial.suggest_float('C', 1e-4, 1e3, log=True)
    
    l1_ratio = None
    if penalty == 'elasticnet':
        l1_ratio = trial.suggest_float('l1_ratio', 0., 1.)
    
    class_weight = trial.suggest_categorical(
        'class_weight', [None, 'balanced']
    )
    
    suggested_estimator_params = {
        "solver": solver,
        "penalty": penalty,
        "C": C,
        "l1_ratio": l1_ratio,
        "class_weight": class_weight,
    }
    estimator_params = update_estimator_params(
        ml_pipe,
        suggested_estimator_params
    )
    
    score = ml_pipe.step(
        estimator=ml_pipe._pipeline_params['estimator'],
        estimator_params=estimator_params,
    )
    
    return score

optimize_estimator_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    params_to_process=params_to_process,
    objective=logreg_objective,
    n_trials=n_trials,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ..\data\df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-18 11:40:01,296] A new study created in memory with name: model_study
[I 2025-04-18 11:40:01,457] Trial 0 finished with value: 0.8101682090919139 and parameters: {'solver': 'liblinear', 'penalty_liblinear': 'l2', 'C': 1.6136341713591311, 'class_weight': None}. Best is trial 0 with value: 0.8101682090919139.
[I 2025-04-18 11:40:01,619] Trial 1 finished with value: 0.8226402532926312 and parameters: {'solver': 'lbfgs', 'penalty_other': 'l2', 'C': 0.47129737561107793, 'class_weight': None}. Best is trial 1 with value: 0.8226402532926312.
[I 2025-04-18 11:40:01,793] Trial 2 finished with value: 0.26121337827326935 and parameters: {'solver': 'saga', 'penalty_saga': 'elasticnet', 'C': 0.00021142332035497166, 'l1_ratio': 0.6075448519014384, 'class_weight': None}. Best is trial 1 with value: 0.8226402532926312.
[I 2025-04-18 11:40:01,955] Trial 3 finished with value: 0.7998648979486092 and parameters: {'solver': 'liblinear', 'penalty_liblinear': 'l1', 'C': 0.292575779498244, 'class_

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "LogisticRegression"


,0
target,splashing
model,LogisticRegression_splashing_smote_opt-smote_o...
holdout_test_f1_macro,0.772036
holdout_test_accuracy_balanced,0.77662
holdout_test_roc_auc,0.885031
holdout_test_f1,0.829787
holdout_test_accuracy,0.786667
cv_test_f1_macro_median,0.807971
cv_test_accuracy_balanced_median,0.832817
cv_test_roc_auc_median,0.877709


Model saved in ..\results\models_modelling4\LogisticRegression_splashing_smote_opt-smote_opt-model


In [7]:
target = 'no_fragmentation'
estimator_params = {
    "fit_intercept": False,
    'max_iter': 1000,
}

optimize_estimator_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    params_to_process=params_to_process,
    objective=logreg_objective,
    n_trials=n_trials,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ..\data\df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-18 11:47:32,728] A new study created in memory with name: model_study
[I 2025-04-18 11:47:32,860] Trial 0 finished with value: 0.8219763971993025 and parameters: {'solver': 'liblinear', 'penalty_liblinear': 'l2', 'C': 1.6136341713591311, 'class_weight': None}. Best is trial 0 with value: 0.8219763971993025.
[I 2025-04-18 11:47:32,994] Trial 1 finished with value: 0.7921977490316808 and parameters: {'solver': 'lbfgs', 'penalty_other': 'l2', 'C': 0.47129737561107793, 'class_weight': None}. Best is trial 0 with value: 0.8219763971993025.
[I 2025-04-18 11:47:33,133] Trial 2 finished with value: 0.4233049487844008 and parameters: {'solver': 'saga', 'penalty_saga': 'elasticnet', 'C': 0.00021142332035497166, 'l1_ratio': 0.6075448519014384, 'class_weight': None}. Best is trial 0 with value: 0.8219763971993025.
[I 2025-04-18 11:47:33,263] Trial 3 finished with value: 0.7954947086214964 and parameters: {'solver': 'liblinear', 'penalty_liblinear': 'l1', 'C': 0.292575779498244, 'class_w

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "LogisticRegression"


,0
target,no_fragmentation
model,LogisticRegression_no_fragmentation_smote_opt-...
holdout_test_f1_macro,0.798595
holdout_test_accuracy_balanced,0.834091
holdout_test_roc_auc,0.949091
holdout_test_f1,0.723404
holdout_test_accuracy,0.826667
cv_test_f1_macro_median,0.84043
cv_test_accuracy_balanced_median,0.874542
cv_test_roc_auc_median,0.93956


Model saved in ..\results\models_modelling4\LogisticRegression_no_fragmentation_smote_opt-smote_opt-model


# KNClassifier

In [8]:
estimator = KNeighborsClassifier
target = 'splashing'
estimator_params = {}
# params_to_process = ['penalty']
params_to_process = None


def knn_objective(trial:optuna.trial.Trial, ml_pipe:MLPipeline):

    n_neighbors = trial.suggest_int("n_neighbors", 1, 50)
    weights = trial.suggest_categorical("weights", ["uniform", "distance"])
    p = trial.suggest_int("p", 1, 2)  # 1=manhattan, 2=euclidean
    # leaf_size - affects the speed of the tree construction and search
    # algorithm - affects the speed of the search, for small datasets it is okay to use auto
    
    suggested_estimator_params = {
        "n_neighbors": n_neighbors,
        "weights": weights,
        "p": p,
    }
    estimator_params = update_estimator_params(
        ml_pipe,
        suggested_estimator_params
    )
    
    score = ml_pipe.step(
        estimator=ml_pipe._pipeline_params['estimator'],
        estimator_params=estimator_params,
    )
    
    return score

optimize_estimator_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    params_to_process=params_to_process,
    objective=knn_objective,
    n_trials=n_trials,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ..\data\df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-18 11:53:59,922] A new study created in memory with name: model_study
[I 2025-04-18 11:54:00,174] Trial 0 finished with value: 0.801645390136896 and parameters: {'n_neighbors': 19, 'weights': 'uniform', 'p': 2}. Best is trial 0 with value: 0.801645390136896.
[I 2025-04-18 11:54:00,413] Trial 1 finished with value: 0.805694867690666 and parameters: {'n_neighbors': 8, 'weights': 'uniform', 'p': 2}. Best is trial 1 with value: 0.805694867690666.
[I 2025-04-18 11:54:00,660] Trial 2 finished with value: 0.785590059640236 and parameters: {'n_neighbors': 31, 'weights': 'uniform', 'p': 2}. Best is trial 1 with value: 0.805694867690666.
[I 2025-04-18 11:54:00,911] Trial 3 finished with value: 0.7591023494245429 and parameters: {'n_neighbors': 42, 'weights': 'uniform', 'p': 1}. Best is trial 1 with value: 0.805694867690666.
[I 2025-04-18 11:54:01,165] Trial 4 finished with value: 0.777523868190684 and parameters: {'n_neighbors': 16, 'weights': 'uniform', 'p': 1}. Best is trial 1 with 

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "KNeighborsClassifier"


,0
target,splashing
model,KNeighborsClassifier_splashing_smote_opt-smote...
holdout_test_f1_macro,0.8333
holdout_test_accuracy_balanced,0.820602
holdout_test_roc_auc,0.820602
holdout_test_f1,0.891089
holdout_test_accuracy,0.853333
cv_test_f1_macro_median,0.896201
cv_test_accuracy_balanced_median,0.891641
cv_test_roc_auc_median,0.891641


Model saved in ..\results\models_modelling4\KNeighborsClassifier_splashing_smote_opt-smote_opt-model


In [9]:
estimator = KNeighborsClassifier
target = 'no_fragmentation'
estimator_params = {}
params_to_process = None

optimize_estimator_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    params_to_process=params_to_process,
    objective=knn_objective,
    n_trials=n_trials,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ..\data\df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-18 12:02:07,219] A new study created in memory with name: model_study
[I 2025-04-18 12:02:07,447] Trial 0 finished with value: 0.7901378337109034 and parameters: {'n_neighbors': 19, 'weights': 'uniform', 'p': 2}. Best is trial 0 with value: 0.7901378337109034.
[I 2025-04-18 12:02:07,654] Trial 1 finished with value: 0.8281425218668367 and parameters: {'n_neighbors': 8, 'weights': 'uniform', 'p': 2}. Best is trial 1 with value: 0.8281425218668367.
[I 2025-04-18 12:02:07,870] Trial 2 finished with value: 0.770901019268706 and parameters: {'n_neighbors': 31, 'weights': 'uniform', 'p': 2}. Best is trial 1 with value: 0.8281425218668367.
[I 2025-04-18 12:02:08,079] Trial 3 finished with value: 0.777411682244389 and parameters: {'n_neighbors': 42, 'weights': 'uniform', 'p': 1}. Best is trial 1 with value: 0.8281425218668367.
[I 2025-04-18 12:02:08,281] Trial 4 finished with value: 0.7755189473072269 and parameters: {'n_neighbors': 16, 'weights': 'uniform', 'p': 1}. Best is trial 1

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "KNeighborsClassifier"


,0
target,no_fragmentation
model,KNeighborsClassifier_no_fragmentation_smote_op...
holdout_test_f1_macro,0.839194
holdout_test_accuracy_balanced,0.861364
holdout_test_roc_auc,0.911364
holdout_test_f1,0.772727
holdout_test_accuracy,0.866667
cv_test_f1_macro_median,0.854396
cv_test_accuracy_balanced_median,0.882051
cv_test_roc_auc_median,0.940476


Model saved in ..\results\models_modelling4\KNeighborsClassifier_no_fragmentation_smote_opt-smote_opt-model


# SVC

In [6]:
estimator = SVC
target = 'splashing'
estimator_params = {
    'probability': True, # Would beFalse for optimization, True for metrics
    # 'shrinking': True, # already in the default params
}
params_to_process = ['gamma']


def svc_objective(trial:optuna.trial.Trial, ml_pipe:MLPipeline):
    kernel = trial.suggest_categorical(
        'kernel', ['linear', 'rbf', 'sigmoid', 'poly']
    )
    C = trial.suggest_float('C', 1e-3, 1e3, log=True)
    
    class_weight = trial.suggest_categorical(
        'class_weight', [None, 'balanced']
    )
    
    suggested_estimator_params = {
        "kernel": kernel,
        "C": C,
        "class_weight": class_weight,
        # Optional params
        # 'gamma': gamma
        # 'coef0': coef0,
        # 'degree': degree,
    }
    
    if kernel in ['rbf', 'poly', 'sigmoid']:
        gamma_choice = trial.suggest_categorical(
            "gamma_type", ["scale", "auto", "float"]
        )
        if gamma_choice == "float":
            gamma = trial.suggest_float("gamma", 1e-4, 10, log=True)
        else:
            gamma = gamma_choice
        suggested_estimator_params['gamma'] = gamma
    
    # Optional params
    if kernel in ['sigmoid', 'poly']:
        suggested_estimator_params['coef0'] = trial.suggest_float(
            "coef0", 0.0, 1.0
        )
    if kernel == 'poly':
        suggested_estimator_params['degree'] = trial.suggest_int(
            "degree", 2, 5
        )
    
    estimator_params = update_estimator_params(
        ml_pipe,
        suggested_estimator_params
    )
    
    # print(estimator_params)
    
    score = ml_pipe.step(
        estimator=ml_pipe._pipeline_params['estimator'],
        estimator_params=estimator_params,
    )
    
    return score

optimize_estimator_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    params_to_process=params_to_process,
    objective=svc_objective,
    n_trials=n_trials,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "splashing" from {'no_fragmentation', 'splashing'}
Load split indexes from: ..\data\df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[I 2025-04-18 14:27:15,550] A new study created in memory with name: model_study
[I 2025-04-18 14:27:15,777] Trial 0 finished with value: 0.392616568979092 and parameters: {'kernel': 'rbf', 'C': 0.008632008168602538, 'class_weight': None, 'gamma_type': 'scale'}. Best is trial 0 with value: 0.392616568979092.
[I 2025-04-18 14:27:16,025] Trial 1 finished with value: 0.3193900243566663 and parameters: {'kernel': 'rbf', 'C': 0.012329623163659839, 'class_weight': 'balanced', 'gamma_type': 'scale'}. Best is trial 0 with value: 0.392616568979092.
[I 2025-04-18 14:27:16,178] Trial 2 finished with value: 0.8434883138156384 and parameters: {'kernel': 'linear', 'C': 0.5450293694558254, 'class_weight': None}. Best is trial 2 with value: 0.8434883138156384.
[I 2025-04-18 14:27:16,374] Trial 3 finished with value: 0.4194501950246622 and parameters: {'kernel': 'poly', 'C': 0.010547383621352036, 'class_weight': 'balanced', 'gamma_type': 'scale', 'coef0': 0.09767211400638387, 'degree': 4}. Best is tria

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

no summary in estimator class "SVC"


,0
target,splashing
model,SVC_splashing_smote_opt-smote_opt-model
holdout_test_f1_macro,0.79
holdout_test_accuracy_balanced,0.78125
holdout_test_roc_auc,0.89429
holdout_test_f1,0.86
holdout_test_accuracy,0.813333
cv_test_f1_macro_median,0.898584
cv_test_accuracy_balanced_median,0.903251
cv_test_roc_auc_median,0.922601


Model saved in ..\results\models_modelling4\SVC_splashing_smote_opt-smote_opt-model


In [6]:
estimator = SVC
target = 'no_fragmentation'
estimator_params = {
    'probability': True, # Would be False for optimization, True for metrics
    # 'shrinking': True, # already in the default params
}
params_to_process = ['gamma']

def svc_objective(trial:optuna.trial.Trial, ml_pipe:MLPipeline):
    kernel = trial.suggest_categorical(
        'kernel', ['linear', 'rbf', 'sigmoid', 'poly']
    )
    C = trial.suggest_float('C', 1e-3, 1e3, log=True)
    
    class_weight = trial.suggest_categorical(
        'class_weight', [None, 'balanced']
    )
    
    suggested_estimator_params = {
        "kernel": kernel,
        "C": C,
        "class_weight": class_weight,
        # Optional params
        # 'gamma': gamma
        # 'coef0': coef0,
        # 'degree': degree,
    }
    
    if kernel in ['rbf', 'poly', 'sigmoid']:
        gamma_choice = trial.suggest_categorical(
            "gamma_type", ["scale", "auto", "float"]
        )
        if gamma_choice == "float":
            gamma = trial.suggest_float("gamma", 1e-4, 10, log=True)
        else:
            gamma = gamma_choice
        suggested_estimator_params['gamma'] = gamma
    
    # Optional params
    if kernel in ['sigmoid', 'poly']:
        suggested_estimator_params['coef0'] = trial.suggest_float(
            "coef0", 0.0, 1.0
        )
    if kernel == 'poly':
        suggested_estimator_params['degree'] = trial.suggest_int(
            "degree", 2, 5
        )
    
    estimator_params = update_estimator_params(
        ml_pipe,
        suggested_estimator_params
    )
    
    # print(estimator_params)
    
    score = ml_pipe.step(
        estimator=ml_pipe._pipeline_params['estimator'],
        estimator_params=estimator_params,
    )
    
    return score

optimize_estimator_optuna(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    params_to_process=params_to_process,
    objective=svc_objective,
    n_trials=500,
    seed=21,
)

Load dataset from: ..\data\df_dimless.xlsx
Keep "no_fragmentation" from {'no_fragmentation', 'splashing'}
Load split indexes from: ..\data\df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[I 2025-04-18 15:04:34,247] A new study created in memory with name: model_study
[I 2025-04-18 15:04:34,411] Trial 0 finished with value: 0.7081493505844338 and parameters: {'kernel': 'sigmoid', 'C': 0.017200322569383943, 'class_weight': 'balanced', 'gamma_type': 'scale', 'coef0': 0.06957095461260054}. Best is trial 0 with value: 0.7081493505844338.
[I 2025-04-18 15:04:34,628] Trial 1 finished with value: 0.8264580342904377 and parameters: {'kernel': 'linear', 'C': 152.1240458771676, 'class_weight': 'balanced'}. Best is trial 1 with value: 0.8264580342904377.
[I 2025-04-18 15:04:34,753] Trial 2 finished with value: 0.7553041972774495 and parameters: {'kernel': 'linear', 'C': 0.04207446825995604, 'class_weight': 'balanced'}. Best is trial 1 with value: 0.8264580342904377.
[I 2025-04-18 15:04:34,892] Trial 3 finished with value: 0.7703212141287935 and parameters: {'kernel': 'linear', 'C': 0.05054268043577859, 'class_weight': 'balanced'}. Best is trial 1 with value: 0.8264580342904377.
[I

best_params


...

('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

no summary in estimator class "SVC"


,0
target,no_fragmentation
model,SVC_no_fragmentation_smote_opt-smote_opt-model
holdout_test_f1_macro,0.829545
holdout_test_accuracy_balanced,0.829545
holdout_test_roc_auc,0.949091
holdout_test_f1,0.75
holdout_test_accuracy,0.866667
cv_test_f1_macro_median,0.90293
cv_test_accuracy_balanced_median,0.90293
cv_test_roc_auc_median,0.956044


Model saved in ..\results\models_modelling4\SVC_no_fragmentation_smote_opt-smote_opt-model
